# 🎮 DQN Agent Training for Atari Pong

This notebook trains a Deep Q-Network (DQN) agent to play Atari Pong.

## Quick Start
1. **Enable GPU**: Settings → Accelerator → GPU T4 x2 or P100
2. **Run all cells**: Run → Run All
3. **Download model**: Use the Output tab to download `final_model.pth`

---

## 📦 Step 1: Install Dependencies

In [ ]:
# Install dependencies
!pip install -q gymnasium[atari,accept-rom-license] shimmy[atari] ale-py
!pip install -q opencv-python torch torchvision

# Verify GPU
import torch
print(f"\n🔧 PyTorch version: {torch.__version__}")
print(f"🎮 GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Device: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: No GPU detected! Enable GPU in Settings → Accelerator")

## 🧠 Step 2: Define Neural Network

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import random
import os

class AtariCNN(nn.Module):
    """Nature DQN CNN - 3 conv layers + 2 FC layers"""
    def __init__(self, action_dim):
        super(AtariCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU()
        )
        self.fc = nn.Sequential(
            nn.Linear(64 * 7 * 7, 512),
            nn.ReLU(),
            nn.Linear(512, action_dim)
        )

    def forward(self, x):
        x = x.float() / 255.0
        x = self.features(x)
        x = x.reshape(x.size(0), -1)
        return self.fc(x)

print("✅ AtariCNN defined")

## 💾 Step 3: Experience Replay Buffer

In [ ]:
class ExperienceReplayBuffer:
    """Circular buffer with pre-allocated numpy arrays"""
    def __init__(self, capacity, state_shape):
        self.capacity = capacity
        self.ptr = 0
        self.size = 0
        
        self.states = np.zeros((capacity, *state_shape), dtype=np.uint8)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.next_states = np.zeros((capacity, *state_shape), dtype=np.uint8)
        self.dones = np.zeros(capacity, dtype=np.bool_)

    def push(self, state, action, reward, next_state, done):
        self.states[self.ptr] = state
        self.actions[self.ptr] = action
        self.rewards[self.ptr] = reward
        self.next_states[self.ptr] = next_state
        self.dones[self.ptr] = done
        self.ptr = (self.ptr + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        indices = np.random.choice(self.size, batch_size, replace=False)
        return (
            torch.from_numpy(self.states[indices]),
            torch.from_numpy(self.actions[indices]),
            torch.from_numpy(self.rewards[indices]),
            torch.from_numpy(self.next_states[indices]),
            torch.from_numpy(self.dones[indices])
        )
    
    def __len__(self):
        return self.size

print("✅ ExperienceReplayBuffer defined")

## 🤖 Step 4: DQN Agent

In [ ]:
class DQNAgent:
    def __init__(self, model_class, action_dim, device, lr=1e-4, gamma=0.99,
                 epsilon_start=1.0, epsilon_final=0.01, epsilon_decay=1000000):
        self.action_dim = action_dim
        self.device = device
        self.gamma = gamma
        
        self.epsilon = epsilon_start
        self.epsilon_final = epsilon_final
        self.epsilon_decay = epsilon_decay
        self.steps_done = 0

        self.policy_net = model_class(action_dim).to(device)
        self.target_net = model_class(action_dim).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)

    def select_action(self, state):
        self.epsilon = max(self.epsilon_final,
                           1.0 - (self.steps_done / self.epsilon_decay))
        self.steps_done += 1

        if random.random() > self.epsilon:
            with torch.no_grad():
                state_t = torch.tensor(state, device=self.device).unsqueeze(0)
                return self.policy_net(state_t).argmax().item()
        else:
            return random.randrange(self.action_dim)

    def update_model(self, experiences):
        states, actions, rewards, next_states, dones = experiences
        
        states = states.to(self.device)
        actions = actions.to(self.device).unsqueeze(1)
        rewards = rewards.to(self.device)
        next_states = next_states.to(self.device)
        dones = dones.to(self.device).float()

        rewards = torch.clamp(rewards, -1.0, 1.0)
        current_q = self.policy_net(states).gather(1, actions)
        avg_q = current_q.mean().item()

        with torch.no_grad():
            max_next_q = self.target_net(next_states).max(1)[0]
            target_q = rewards + (self.gamma * max_next_q * (1 - dones))

        loss = F.smooth_l1_loss(current_q.squeeze(), target_q)

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 1.0)
        self.optimizer.step()

        return loss.item(), avg_q

    def sync_target_network(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())

print("✅ DQNAgent defined")

## 🎮 Step 5: Environment Wrappers

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import cv2
import ale_py

gym.register_envs(ale_py)

class AtariPreprocessing(gym.Wrapper):
    def __init__(self, env, screen_size=84):
        super().__init__(env)
        self.screen_size = screen_size
        self.observation_space = spaces.Box(
            low=0, high=255, shape=(screen_size, screen_size, 1), dtype=np.uint8
        )

    def step(self, action):
        total_reward = 0.0
        terminated = truncated = False
        frame_buffer = np.zeros((2, 210, 160, 3), dtype=np.uint8)
        
        for i in range(4):
            obs, reward, term, trunc, info = self.env.step(action)
            if i >= 2:
                frame_buffer[i - 2] = obs
            total_reward += float(reward)
            terminated = terminated or term
            truncated = truncated or trunc
            if terminated or truncated:
                break
        
        max_frame = frame_buffer.max(axis=0)
        return self._preprocess(max_frame), total_reward, terminated, truncated, info

    def _preprocess(self, frame):
        img = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        img = cv2.resize(img, (self.screen_size, self.screen_size))
        return img[:, :, np.newaxis]

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        return self._preprocess(obs), info


class FrameStack(gym.Wrapper):
    def __init__(self, env, k=4):
        super().__init__(env)
        self.k = k
        self.frames = []
        shp = env.observation_space.shape
        self.observation_space = spaces.Box(
            low=0, high=255, shape=(k, shp[0], shp[1]), dtype=np.uint8
        )

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.frames = [obs for _ in range(self.k)]
        return self._get_ob(), info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        self.frames.pop(0)
        self.frames.append(obs)
        return self._get_ob(), reward, terminated, truncated, info

    def _get_ob(self):
        return np.stack(self.frames, axis=0).squeeze()


def make_atari_env(game_id):
    env = gym.make(game_id)
    env = AtariPreprocessing(env)
    env = FrameStack(env, k=4)
    return env

print("✅ Environment wrappers defined")

## 🔄 Step 6: Check for Existing Checkpoint

In [ ]:
import glob

# Kaggle uses /kaggle/working for outputs
SAVE_DIR = '/kaggle/working'
os.makedirs(SAVE_DIR, exist_ok=True)

def find_latest_checkpoint(save_dir):
    """Find the most recent checkpoint in the save directory."""
    checkpoints = glob.glob(os.path.join(save_dir, 'checkpoint_*.pth'))
    if not checkpoints:
        return None, 0
    
    latest = max(checkpoints, key=lambda x: int(x.split('_')[-1].replace('.pth', '')))
    step = int(latest.split('_')[-1].replace('.pth', ''))
    return latest, step

latest_checkpoint, resume_step = find_latest_checkpoint(SAVE_DIR)

if latest_checkpoint:
    print(f"🔄 Found checkpoint: {latest_checkpoint}")
    print(f"   Training will resume from step {resume_step:,}")
else:
    print("🆕 No checkpoint found. Training will start from scratch.")

## 🚀 Step 7: Training Loop

In [ ]:
from collections import deque
import time

def train(save_dir, resume_from=None):
    # ========== Configuration ==========
    GAME_ID = "ALE/Pong-v5"
    BATCH_SIZE = 32
    LR = 1e-4
    GAMMA = 0.99
    REPLAY_SIZE = 100000
    TARGET_UPDATE_FREQ = 10000
    TOTAL_STEPS = 3000000  # 3M steps for better results
    CHECKPOINT_FREQ = 50000
    EARLY_STOP_REWARD = 10.0
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🎮 Training on: {device}")
    
    env = make_atari_env(GAME_ID)
    agent = DQNAgent(AtariCNN, env.action_space.n, device, lr=LR, gamma=GAMMA)
    buffer = ExperienceReplayBuffer(REPLAY_SIZE, env.observation_space.shape)
    
    start_step = 1
    
    # Resume from checkpoint if available
    if resume_from and os.path.exists(resume_from):
        print(f"\n🔄 Loading checkpoint: {resume_from}")
        checkpoint = torch.load(resume_from, map_location=device)
        agent.policy_net.load_state_dict(checkpoint['model_state_dict'])
        agent.target_net.load_state_dict(checkpoint['model_state_dict'])
        agent.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        agent.epsilon = checkpoint['epsilon']
        agent.steps_done = checkpoint.get('steps_done', checkpoint['step'])
        start_step = checkpoint['step'] + 1
        print(f"✅ Resumed from step {start_step:,}, epsilon={agent.epsilon:.4f}")
    
    state, _ = env.reset()
    episode_rewards = deque(maxlen=100)
    current_episode_reward = 0
    start_time = time.time()
    
    print(f"\n{'='*50}")
    print(f"🚀 Training started! Target: avg reward ≥ {EARLY_STOP_REWARD}")
    print(f"{'='*50}\n")
    
    for step in range(start_step, TOTAL_STEPS + 1):
        action = agent.select_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        current_episode_reward += reward
        
        buffer.push(state, action, reward, next_state, done)
        state = next_state
        
        if len(buffer) > BATCH_SIZE:
            experiences = buffer.sample(BATCH_SIZE)
            loss, _ = agent.update_model(experiences)
        
        if step % TARGET_UPDATE_FREQ == 0:
            agent.sync_target_network()
        
        if done:
            episode_rewards.append(current_episode_reward)
            avg_reward = np.mean(episode_rewards)
            
            if len(episode_rewards) % 10 == 0:
                elapsed = (time.time() - start_time) / 60
                print(f"Step {step:>8,} | Avg Reward: {avg_reward:>6.1f} | "
                      f"Epsilon: {agent.epsilon:.3f} | Time: {elapsed:.1f}min")
            
            if len(episode_rewards) == 100 and avg_reward >= EARLY_STOP_REWARD:
                print(f"\n🎉 GOAL REACHED! Average reward: {avg_reward:.1f}")
                final_path = os.path.join(save_dir, 'final_model.pth')
                torch.save({
                    'step': step,
                    'model_state_dict': agent.policy_net.state_dict(),
                    'optimizer_state_dict': agent.optimizer.state_dict(),
                    'epsilon': agent.epsilon,
                    'steps_done': agent.steps_done
                }, final_path)
                print(f"💾 Final model saved to: {final_path}")
                return final_path
            
            state, _ = env.reset()
            current_episode_reward = 0
        
        if step % CHECKPOINT_FREQ == 0:
            checkpoint_path = os.path.join(save_dir, f'checkpoint_{step}.pth')
            torch.save({
                'step': step,
                'model_state_dict': agent.policy_net.state_dict(),
                'optimizer_state_dict': agent.optimizer.state_dict(),
                'epsilon': agent.epsilon,
                'steps_done': agent.steps_done
            }, checkpoint_path)
            print(f"💾 Checkpoint saved: {checkpoint_path}")
    
    # Save final model
    final_path = os.path.join(save_dir, 'final_model.pth')
    torch.save({
        'step': step,
        'model_state_dict': agent.policy_net.state_dict(),
        'optimizer_state_dict': agent.optimizer.state_dict(),
        'epsilon': agent.epsilon,
        'steps_done': agent.steps_done
    }, final_path)
    
    print(f"\n{'='*50}")
    print(f"✅ Training complete!")
    print(f"   Final avg reward: {np.mean(episode_rewards):.1f}")
    print(f"   Model saved to: {final_path}")
    print(f"{'='*50}")
    
    return final_path

print("✅ Training function defined!")

## 🎮 Step 8: START TRAINING!

In [ ]:
# 🎮 START TRAINING!
final_model_path = train(SAVE_DIR, resume_from=latest_checkpoint)

## 📥 Step 9: View Output Files

After training, find your model in the **Output** tab on the right panel.
- `final_model.pth` - The trained model
- `checkpoint_*.pth` - Intermediate checkpoints

In [ ]:
# List saved files
import os
print("\n📁 Saved files in /kaggle/working:")
for f in os.listdir('/kaggle/working'):
    if f.endswith('.pth'):
        size_mb = os.path.getsize(f'/kaggle/working/{f}') / (1024*1024)
        print(f"  📄 {f} ({size_mb:.1f} MB)")

---

## 🔧 Troubleshooting

**If no GPU:**
1. Go to Settings → Accelerator → GPU T4 x2
2. Click "Save" and re-run all cells

**If kernel restarts:**
- Checkpoints are auto-saved every 50k steps
- Just re-run all cells to resume from last checkpoint

**Kaggle Session Limits:**
- GPU sessions: 30 hours/week
- Max runtime: 12 hours per session